# 03_01_Building Dimension Station

In [1]:
from pathlib import Path

import geopandas as gpd
import pandas as pd
from shapely import wkb

import infrabel_punctuality as ip

In [2]:
PATH_INTERMEDIATE = Path("../data/intermediate/")
PATH_PROCESSED = Path("../data/processed/")

In [3]:
METRICS_CRS_GEOMETRY = 4326

# Table of Contents
- [1. DIMENSION STATION](#1-dimension-station)

- [1.1. Loading Datasets](#11-loading-datasets)

- [1.2. Merging `municipalities` and `population`](#12-merging-municipalities-and-population)

- [1.3. Processing Geographical Data](#13-processing-geographical-data)
    - [1.3.1. Reprojecting `territorial_divisions` to EPSG:4326](#131-reprojecting-territorial_divisions-to-epsg4326)
    - [1.3.2. Merging `territorial_divisions_4326` and `municipalities_population`](#132-merging-territorial_divisions_4326-and-municipalities_population)
    - [1.3.3. Resolving Statbel Reference Dataset Desynchronisation](#133-resolving-statbel-reference-dataset-desynchronisation)
    - [1.3.4. Preparing `stations` for Spatial Join](#134-preparing-stations-for-spatial-join)
    - [1.3.5. Joining Spatially `municipalities_geo` and `stations`](#135-joining-spatially-municipalities_geo-and-stations)
    - [1.3.6. Extracting `lon` and `lat` from `geometry`](#136-extracting-lon-and-lat-from-geometry)

- [1.4. Refining `Dim_Station`](#14-refining-dim_station)
    - [1.4.1. Binning `Population_density` into `Population_density_category`](#141-binning-population_density-into-population_density_category)
    - [1.4.2. Adding Orphan Station Entries Manually](#142-adding-orphan-station-entries-manually)
    - [1.4.3. Reorganising Column Order](#143-reorganising-column-order)

- [1.5. Exporting to Gold Layer](#15-exporting-to-gold-layer)


# 1. DIMENSION STATION

**DESCRIPTION**

`Dim_Station` results from the spatial join between `stations_cleaned` and `territorial_divisions_3812_prepared`, enriched by `municipalities_flattened` and `population_cleaned`.

Its **Business Key** is `Ptcar_ID`. Even though it is a stable business key provided by Infrabel, a **Surrogate Key** is added to the dimension to preserve the history of changes.

Once the dimension is loaded into SQL, this Surrogate Key will serve as the **Foreign Key** linking `Dim_Station` to `Fact_Punctuality`.

Its **grain** is one row for each station, or active passenger stop of any sort.

It contains **799 entries**: 794 operational points extracted from the Infrabel station reference dataset, plus 5 orphan stations manually added to ensure Foreign Key integrity with `Fact_Punctuality` (see **NOTE** below). This figure intentionally exceeds the 554 commercial stations officially reported by the SNCB, as Infrabel's infrastructure-based definition of a passenger stop is broader than SNCB's commercial definition, but it includes all passenger stops present in `Fact_punctuality`.

In the star schema data warehouse, `Dim_Station` enables punctuality analyses by **station**, by Belgian administrative entities such as **municipalities, provinces, or regions**, and by **population density category**. 

Associated with the `territorial_divisions_4326` layer background loaded to Power BI, it also enables the **cartographic representation** of all Infrabel active stations across Belgium.

<br>

**ATTRIBUTES**

`Dim_Station` has **18 attributes**:

* `SK_Station` (Int64): Dimension Surrogate Key.

* `Ptcar_ID` (Int64): station IDs (Business Key)

* `Station_Name_French` and `Station_Name_Dutch` (string): Infrabel station names in French and Dutch 

* `Station_Class` (string): Infrabel classes for operational points of the railway network served by passenger trains (i.e. `"Station"`, `"Stop in open track"`, `"Service installation"`, `"Service stop"`, and `"Grid"`)

* `Station_Status` (string): stations flagged as `"unmatched"` or `"active"` (see **NOTE** below)

* `REFNIS_Municipality` (Int64): municipality REFNIS codes

* `Communes` and `Gemeenten` (string): French and Dutch names of the municipalities where the stations are located

* `REFNIS_Province` (Int64): province REFNIS codes

* `Provinces` and `Provincies` (string): French and Dutch names of the provinces where the stations are located

* `REFNIS_Region` (Int64): region REFNIS codes

*  `Regions` and `Gewesten` (string): French and Dutch names of the regions where the stations are located

* `Population_Density_Category` (string): population density categories derived from the `Float64` column `Population_density`, which is dropped after computation (three classes: `"cities"`, `"towns_suburbs"`, `"rural"`)

* `lon` (Float64): longitude coordinate of each station

* `lat` (Float64): latitude coordinate of each station

<br>

**SOURCES**

The raw source datasets are:
* For `stations_cleaned`: `operational_pts_railway.parquet` - the Infrabel reference dataset for stations and passenger stops.
* For `territorial_divisions_3812_prepared`: `territorialdivisions_3812.gpkg` - the Geo.be geopackage that includes geographic coordinates for all Belgian territorial divisions, selected to the municipality layer.
* For `municipalities_flattened`: `communes.xlsx` - the Statbel reference dataset for Belgian municipalities, provinces, and regions.
* For `population_cleaned`: `population.xlsx` - the Statbel reference dataset for Belgian population statistics sorted by municipalities.

**BUILDING PROCESS**

1) Datasets are imported: 
    - DataFrame `municipalities` from `municipalities_flattened`
    - DataFrame `population` from `population_cleaned`
    - DataFrame `stations` from `stations_cleaned`
    - GeoDataFrame from `territorial_divisions_3812_prepared` (**Section 1.1.**)

2) `municipalities` and `population` are merged on their REFNIS code columns. This merge creates the `municipalities_population` DataFrame (**Section 1.2.**).

3) The Coordinate Reference System of `territorial_divisions` is reprojected from the Belgian Lambert projection `EPSG:3812` to the geographic coordinate system *World Geodetic System 1984* `EPSG:4326` to enable the spatial join with `stations`, and ensure compatibility with Power BI maps. From this operation, the GeoDataFrame `territorial_divisions_4326` is created (**Section 1.3.1.**).

4) `territorial_divisions_4326` and `municipalities_population` are merged on their REFNIS code columns. This merge creates the `geo_municipalities` GeoDataFrame (**Section 1.3.2.**).

5) **Desynchronised Statbel reference datasets**: unlike the Statbel `communes` dataset, the Statbel `population` dataset appears **to not have been updated after the latest Belgian municipalities merged on 1 January 2025**. In addition, some recently reassigned REFNIS codes are not updated either (e.g. `82003` for Bastogne exists in the dataset but `82039` for Bastogne does not). Therefore, there are 13 entries in `geo_municipalities` with missing values in their corresponding `population` column: 

    * For 6 out of 13, the values are **recovered via the older REFNIS code**. 

    * For 6 out of 13, the values are calculated by aggregating the `population` values of the old municipalities merged in new ones, calling the `find_population_()` function from the `infrabel_punctuality` package, as the **new municipality names are a concatenation of the old municipality names** (e.g. **Nazareth** and **De Pinte** municipalities merged `Nazareth-La Pinte` on 1 January 2025). 
    
    * The last case is handled manually, as the new municipality name is entirely different from the merged entities: **Gammerages**, **Gooik**, **Herne** municipaliies were merged into `Pajottegem` on 1 January 2025 (**Section 1.3.3.**).

6) `stations` is prepared for the spatial join, parsing WKB (Well-Known Binary) bytes from the `geo_point_2d` column into Shapely points with `shapely`, then transforming the `stations` DataFrame to a GeoDataFrame using the CRS `EPSG:4326` (**Section 1.3.4.**).

7) A spatial left join between `stations` and `geo_municipalities` is operated in order to locate each station in its corresponding municipality (`predicate="within"`). **100%** of the stations match. This spatial join creates the `geo_station` GeoDataFrame (**Section 1.3.5.**).

8) The `lon` and `lat` columns are extracted from the `geometry` of `geo_stations` (**Section 1.3.6.**).

9) A `Population_density_category` column is derived from `population_density` to enable analyses filtered by geographical categories such as `"cities"`, `"towns_suburbs"`, and `"rural"` (**Section 1.4.1.**).

10) A `Station_status` column is added. All entries contain the fixed `"active"` value, except for five orphan stations that are manually added and flagged as `"unmatched"` - see **NOTE** below (**Section 1.4.2.**).

11) Column order is reorganised to improve clarity (**Section 1.4.3.**).

12) `Dim_Station` is exported to the `processed` directory. The result of the export is verified (**Section 1.5.**).

**NOTE - Orphan Stations**:
As established in the *02_04_profiling_and_cleaning_punctuality* notebook, the Infrabel `punctuality` dataset contains five orphan stations which have station IDs (`ptcarid`) and are active in the `punctuality` dataset, but are absent from the Infrabel station reference dataset. To ensure the integrity of the Foreign Key between `Dim_Station` and `Fact_Punctuality`, these stations are manually added to the dimension with the following characteristics:

| French and Dutch Station Name             | ptcarid    | Class_en                      |
|-------------------------------------------|------------|-------------------------------|
| BAULERS                                   | 125        | "Service installation"        |
| MORTSEL (former MORTSEL-DEURNESTEENWEG)   | 864        | "Stop in open track"          |
| OLEN-SAS                                  | 2222       | "Station"                     |
| ANTWERPEN-LINKEROEVER                     | 2291       | "Stop in open track"          |
| BAASRODE-WIJKSPOREN                       | 230        | "Unknown"                     |

Their geographic coordinates are `None`.

## 1.1. Loading Datasets

In [4]:
municipalities = pd.read_parquet(PATH_INTERMEDIATE / "municipalities_flattened.parquet")

In [5]:
stations = pd.read_parquet(PATH_INTERMEDIATE / "stations_cleaned.parquet")

In [6]:
population = pd.read_parquet(PATH_INTERMEDIATE / "population_cleaned.parquet")

In [7]:
territorial_divisions = gpd.read_file(PATH_INTERMEDIATE / "territorial_divisions_3812_prepared.gpkg")

## 1.2. Merging `municipalities` and `population`

* The French and Dutch column names are retained to enable filtering by administrative entities in both languages.

* To enable missing values mapping to `NULL` in the SQL data warehouse after the loading of the dimension via SQLAlchemy, the `str`, `int64`, `float64` columns are cast to `string`, `Int64`, and `Float64`.

In [8]:
municipalities_population = municipalities.merge(
    population,
    left_on="CD_REFNIS_mun",
    right_on="refnis",
    how="left"
)

In [9]:
municipalities_population.info()

<class 'pandas.DataFrame'>
RangeIndex: 2715 entries, 0 to 2714
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   CD_REFNIS_mun       2715 non-null   int64  
 1   Communes            2715 non-null   str    
 2   Gemeenten           2715 non-null   str    
 3   CD_REFNIS_prov      2715 non-null   int64  
 4   Provinces           2715 non-null   str    
 5   Provincies          2715 non-null   str    
 6   CD_REFNIS_region    2715 non-null   int64  
 7   Régions             2715 non-null   str    
 8   Gewesten            2715 non-null   str    
 9   refnis              581 non-null    float64
 10  place_of_residence  581 non-null    str    
 11  population          581 non-null    float64
 12  area                581 non-null    float64
 13  population_density  581 non-null    float64
dtypes: float64(4), int64(3), str(7)
memory usage: 540.2 KB


In [10]:
municipalities_population = municipalities_population.rename(columns={
    "Régions" : "Regions",
    "population_density" : "Population_density"
    })

In [11]:
municipalities_population = municipalities_population.astype({
    "CD_REFNIS_mun" : "Int64",
    "Communes" : "string",
    "Gemeenten" : "string",
    "CD_REFNIS_prov" : "Int64",
    "Provinces" : "string",
    "Provincies" : "string",
    "CD_REFNIS_region" : "Int64",
    "Regions" : "string",
    "Gewesten" : "string",
    "Population_density" : "Float64"
})

## 1.3. Processing Geographical Data

### 1.3.1. Reprojecting `territorial_divisions` to EPSG:4326

* `territorial_divisions` CRS `EPSG:3812` is reprojected to `EPSG:4326`.

In [12]:
territorial_divisions_4326 = territorial_divisions.to_crs(epsg=METRICS_CRS_GEOMETRY)
territorial_divisions_4326.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

### 1.3.2. Merging `territorial_divisions_4326` and `municipalities_population`

* Irrelevant columns are dropped after the merge.

In [13]:
municipalities_geo = territorial_divisions_4326.merge(
    municipalities_population,
    how="left",
    left_on="CD_REFNIS",
    right_on="CD_REFNIS_mun"
)


In [14]:
municipalities_geo.columns

Index(['tgid', 'CD_REFNIS', 'languagestatute', 'municipality_name_fre',
       'municipality_name_dut', 'geometry', 'CD_REFNIS_mun', 'Communes',
       'Gemeenten', 'CD_REFNIS_prov', 'Provinces', 'Provincies',
       'CD_REFNIS_region', 'Regions', 'Gewesten', 'refnis',
       'place_of_residence', 'population', 'area', 'Population_density'],
      dtype='str')

In [15]:
municipalities_geo = municipalities_geo.drop(columns=["CD_REFNIS", 
                                                      "municipality_name_fre", 
                                                      "municipality_name_dut"])

In [16]:
municipalities_geo.head()

,tgid,languagestatute,geometry,CD_REFNIS_mun,Communes,Gemeenten,CD_REFNIS_prov,Provinces,Provincies,CD_REFNIS_region,Regions,Gewesten,refnis,place_of_residence,population,area,Population_density
0,{8BF44CB0-B8FD-44F6-A64F-1307610DA4C9},1,"MULTIPOLYGON Z (((5.58659 51.11022 0, 5.58653 ...",72004,Bree,Bree,70000,Province du Limbourg,Provincie Limburg,2000,Région flamande,Vlaams Gewest,72004.0,Bree,16085.0,64.99,247.52
1,{54A85359-4967-4318-AA63-D234DDED2FD7},2,"MULTIPOLYGON Z (((5.94888 50.62314 0, 5.94886 ...",63004,Baelen,Baelen,60000,Province de Liège,Provincie Luik,3000,Région wallonne,Waals Gewest,63004.0,Baelen,4454.0,85.55,52.06
2,{95E2AAE2-F9DB-456C-A113-2A21ED6F932F},1,"MULTIPOLYGON Z (((5.20479 51.13968 0, 5.20475 ...",13003,Balen,Balen,10000,Province d’Anvers,Provincie Antwerpen,2000,Région flamande,Vlaams Gewest,13003.0,Balen,22618.0,72.99,309.9
3,{4487B4B8-4422-4856-97C5-33174BF84028},2,"MULTIPOLYGON Z (((5.59775 50.74958 0, 5.59769 ...",62011,Bassenge,Bitsingen,60000,Province de Liège,Provincie Luik,3000,Région wallonne,Waals Gewest,62011.0,Bassenge,8905.0,38.21,233.03
4,{68A224E9-B2E7-4225-9FDA-03AE2B9C8C41},2,"MULTIPOLYGON Z (((5.54749 49.70456 0, 5.54587 ...",85046,Habay,Habay,80000,Province du Luxembourg,Provincie Luxemburg,3000,Région wallonne,Waals Gewest,85046.0,Habay,8409.0,103.53,81.22


In [17]:
print(municipalities_geo["geometry"].duplicated().sum())
print(municipalities_geo["CD_REFNIS_mun"].duplicated().sum())
municipalities_geo.duplicated().sum()

0
0


np.int64(0)

### 1.3.3. Resolving Statbel Reference Dataset Desynchronisation

* **13 entries** in `municipalities_geo` are unmatched against `population` and therefore have missing values in the columns imported from `population`  

* 6 out of 13 are municipalities with two REFNIS codes in `municipalities` but only one in `population`. Their population column values are copied from their corresponding `municipalities_geo` entries with the second REFNIS code.

* 7 out of 13 are recent municipal fusions (e.g. the fusion of **Nazareth** and **De Pinte** municipalities in the new `Nazareth-De Pinte` municipality on 1 January 2025) that have not been updated in `population`: 
    * 6 are resolved by calling `find_population_values()` from the `infrabel_package`.
    * One - `Pajottegem` - is resolved manually, as the old municipality names merged have no similarities with the new one (**Gammerages**, **Gooik**, **Herne** were merged into **Pajottegem** on 1 January 2025).

* At the end of this section, the irrelevant columns remaining from the merge are dropped.

In [18]:
municipalities_geo.isnull().sum()

tgid                   0
languagestatute        0
geometry               0
CD_REFNIS_mun          0
Communes               0
Gemeenten              0
CD_REFNIS_prov         0
Provinces              0
Provincies             0
CD_REFNIS_region       0
Regions                0
Gewesten               0
refnis                13
place_of_residence    13
population            13
area                  13
Population_density    13
dtype: int64

In [19]:
nan_rows_from_population = (
    municipalities_geo["Communes"].loc[municipalities_geo["Population_density"].isnull()].to_list()
)
print(f"There are {len(nan_rows_from_population)} municipalities_geo entries with no match in population.")
nan_rows_from_population

There are 13 municipalities_geo entries with no match in population.


['Bastogne',
 'Pajottegem',
 'Wingene',
 'Tielt (Tielt)',
 'Nazareth-De Pinte',
 'Lochristi',
 'Lokeren',
 'Merelbeke-Melle',
 'Beveren-Kruibeke-Zwijndrecht',
 'Bilzen-Hoeselt',
 'Tongres-Looz',
 'Tessenderlo-Ham',
 'Hasselt']

* From these 13 entries, 6 exist in `population` under older REFNIS codes.

* These older REFNIS codes still exist in `municipalities_geo` alongside the new ones.

* The `population` values corresponding to the older REFNIS code entries are assigned to the new ones through a merge on the name columns (`Communes` for `municipalities_geo` and `place_of_residence` for `population`).

In [20]:
old_refnis_entries = (population[["refnis", "place_of_residence"]]
 .loc[population["place_of_residence"]
      .isin(nan_rows_from_population)])
old_refnis_entries

,refnis,place_of_residence
257,37015,Tielt (Tielt)
259,37018,Wingene
303,44034,Lochristi
329,46014,Lokeren
508,71022,Hasselt
554,82003,Bastogne


In [21]:
old_refnis_entry_names = old_refnis_entries["place_of_residence"].to_list()

(municipalities[["CD_REFNIS_mun", "Communes"]].loc[municipalities["Communes"]
                                                   .isin(old_refnis_entry_names)]
                                                   .sort_values("Communes")
)

,CD_REFNIS_mun,Communes
2147,82003,Bastogne
2183,82039,Bastogne
1920,71022,Hasselt
1969,71072,Hasselt
935,44034,Lochristi
986,44087,Lochristi
1060,46014,Lokeren
1075,46029,Lokeren
740,37015,Tielt (Tielt)
747,37022,Tielt (Tielt)


In [22]:
mask_unmatched_population = municipalities_geo["place_of_residence"].isnull()

In [23]:
municipality_name_merge = municipalities_geo[mask_unmatched_population].merge(
    population,
    left_on="Communes",
    right_on="place_of_residence",
    how="left"
)

In [24]:
new_values = municipality_name_merge[[
    "refnis_y", 
    "place_of_residence_y", 
    "population_y", 
    "area_y", 
    "population_density"
]].values

municipalities_geo.loc[
    mask_unmatched_population,
    ["refnis", "place_of_residence", "population", "area", "Population_density"]
] = new_values

* There are still 7 `municipalities_geo` entries with no match in `population`.  

In [25]:
refnis_new_municipalities = (municipalities_geo[["CD_REFNIS_mun", "Communes"]]
                             .loc[municipalities_geo["place_of_residence"].isnull()])
refnis_new_municipalities

,CD_REFNIS_mun,Communes
553,23106,Pajottegem
556,44086,Nazareth-De Pinte
559,44088,Merelbeke-Melle
560,46030,Beveren-Kruibeke-Zwijndrecht
561,73110,Bilzen-Hoeselt
562,73111,Tongres-Looz
563,71071,Tessenderlo-Ham


* After verification, these entries correspond to new municipalities recently merged from older ones (e.g. **Nazareth** and **De Pinte** merged into `Nazareth-De Pinte` on 1 January 2025).

* **Gammerages**, **Gooik**, **Herne** were merged into `Pajottegem` on 1 January 2025. Since the name of this municipality does not match any name of the older municipalities from which it was merged, its case is handled manually before calling the `find_population_values()` function.

In [26]:
#Pajottegem
pajottegem_composantes = (population.loc[
    population["place_of_residence"].isin(["Gammerages", "Gooik", "Herne"])]
)
pajottegem_composantes

,refnis,place_of_residence,population,area,population_density
104,23023,Gammerages,8746,35.29,247.83
105,23024,Gooik,9170,40.21,228.08
108,23032,Herne,6665,44.66,149.24


In [27]:
pajottegem_aggr = pajottegem_composantes.agg({
    "population" : "sum",
    "area" : "sum"
}).to_frame().T

pajottegem_aggr["Population_density"] = pajottegem_aggr["population"] / pajottegem_aggr["area"]

pajottegem_aggr["refnis"] = 23106
pajottegem_aggr["place_of_residence"] = "Pajottegem"

pajottegem_aggr

,population,area,Population_density,refnis,place_of_residence
0,24581.0,120.16,204.568908,23106,Pajottegem


In [28]:
pajottegem_values = pajottegem_aggr[[
    "refnis",
    "place_of_residence",
    "population",
    "area",
    "Population_density"
]].values

mask_pajottegem = (municipalities_geo["CD_REFNIS_mun"].isin(pajottegem_aggr["refnis"]))

cols = ["refnis", "place_of_residence", "population", "area", "Population_density"]

municipalities_geo.loc[mask_pajottegem, cols] = pajottegem_values

* A list of the 6 last `municipalities_geo` entries with no match against `population` is created.

* From this list, `find_population_values()` retrieves the old municipality names and values, then aggregates these values into new entries. 

* These entries are concatenated into a new DataFrame that is then merged with `municipalities_geo`.

In [29]:
new_municipality_names = refnis_new_municipalities["Communes"].to_list()
new_municipality_names.remove("Pajottegem")
new_municipality_names

['Nazareth-De Pinte',
 'Merelbeke-Melle',
 'Beveren-Kruibeke-Zwijndrecht',
 'Bilzen-Hoeselt',
 'Tongres-Looz',
 'Tessenderlo-Ham']

In [30]:
new_municipalities = ip.find_population_values(
    df_population=population,
    df_municipalities=municipalities_geo,
    list_entries=new_municipality_names
)
new_municipalities

,refnis,place_of_residence,population,area,Population_density
0,44086,Nazareth-De Pinte,22302,53.21,419.131742
1,44088,Merelbeke-Melle,36396,52.34,695.376385
2,46030,Beveren-Kruibeke-Zwijndrecht,35801,53.77,665.817370
3,73110,Bilzen-Hoeselt,42016,106.07,396.115773
4,73111,Tongres-Looz,41902,139.13,301.171566
5,71071,Tessenderlo-Ham,29510,84.43,349.520313


In [31]:
new_municipalities_values = new_municipalities.values

mask_new_mun = municipalities_geo["CD_REFNIS_mun"].isin(new_municipalities["refnis"])

cols = ["refnis", "place_of_residence", "population", "area", "Population_density"]

municipalities_geo.loc[mask_new_mun, cols] = new_municipalities_values

In [32]:
municipalities_geo.isnull().sum()

tgid                  0
languagestatute       0
geometry              0
CD_REFNIS_mun         0
Communes              0
Gemeenten             0
CD_REFNIS_prov        0
Provinces             0
Provincies            0
CD_REFNIS_region      0
Regions               0
Gewesten              0
refnis                0
place_of_residence    0
population            0
area                  0
Population_density    0
dtype: int64

* There are no more missing values in `municipalities_geo`.

* As `Population_density` is the only column required for the dimension building, the remaining columns from `population` are dropped.

In [33]:
municipalities_geo = municipalities_geo.drop(columns=["refnis", 
                                                      "place_of_residence", 
                                                      "population", 
                                                      "area"
                                                      ])

### 1.3.4. Preparing `stations` for Spatial Join

* The `geo_point_2d` column is cast as a `geometry` column.

* `stations` is transformed into a GeoDataFrame.

In [34]:
stations.head()

,geo_point_2d,geo_shape,ptcarid,longnamefrench,longnamedutch,class_en
0,b'\x01\x01\x00\x00\x00\x13\x11\xbf\xc37\xfd\x1...,b'\x01\x01\x00\x00\x00\x13\x11\xbf\xc37\xfd\x1...,443,GENK-ZUID-L.O.-FORD,GENK-ZUID-L.O.-FORD,Station
1,b'\x01\x01\x00\x00\x00\r\to\xbe\xd6\xd5\x11@\x...,b'\x01\x01\x00\x00\x00\r\to\xbe\xd6\xd5\x11@\x...,275,CHARLEROI-A.T.,CHARLEROI-A.T.,Service installation
2,b'\x01\x01\x00\x00\x00\xbe\xe17\\7\xc0\r@\x80\...,b'\x01\x01\x00\x00\x00\xbe\xe17\\7\xc0\r@\x80\...,1888,WONDELGEM-BUNDEL,WONDELGEM-BUNDEL,Station
3,b'\x01\x01\x00\x00\x00t\x87S\x00\x96\xf8\t@\xd...,b'\x01\x01\x00\x00\x00t\x87S\x00\x96\xf8\t@\xd...,2155,ZEEBRUGGE-BRAM-TANKINST.,ZEEBRUGGE-BRAM-TANKINST.,Service installation
4,b'\x01\x01\x00\x00\x00\xf0\xdc\xd0t~\xfe\t@\x7...,b'\x01\x01\x00\x00\x00\xf0\xdc\xd0t~\xfe\t@\x7...,2154,ZEEBRUGGE-BRAM-DSP880,ZEEBRUGGE-BRAM-DSP880,Station


In [35]:
# Parse WKB bytes into Shapely Point geometries
stations["geometry"] = stations["geo_point_2d"].apply(wkb.loads)

In [36]:
# Convert to GeoDataFrame in EPSG:4326
stations = gpd.GeoDataFrame(stations, geometry="geometry", crs=METRICS_CRS_GEOMETRY)

In [37]:
print(stations.crs)
print(stations["geometry"].iloc[0])

EPSG:4326
POINT (5.497283037697588 50.92336349484445)


### 1.3.5. Joining Spatially `municipalities_geo` and `stations`

* The spatial left join between `stations` and `municipalities_geo` GeoDataFrames has a match rate of **100%**.

* The resulting GeoDataFrame is named `geo_stations`.

* Irrelevant columns are dropped after the merge.

* Columns are renamed to improve clarity.

In [38]:
geo_stations = gpd.sjoin(
    stations,
    municipalities_geo,
    how="left",
    predicate="within"
)

In [39]:
unmatched = geo_stations[geo_stations["CD_REFNIS_mun"].isnull()]
print(f"Unmatched stations: {len(unmatched)}")

Unmatched stations: 0


In [40]:
geo_stations.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 794 entries, 0 to 793
Data columns (total 20 columns):
 #   Column              Non-Null Count  Dtype   
---  ------              --------------  -----   
 0   geo_point_2d        794 non-null    object  
 1   geo_shape           794 non-null    object  
 2   ptcarid             794 non-null    int64   
 3   longnamefrench      794 non-null    str     
 4   longnamedutch       794 non-null    str     
 5   class_en            794 non-null    str     
 6   geometry            794 non-null    geometry
 7   index_right         794 non-null    int64   
 8   tgid                794 non-null    str     
 9   languagestatute     794 non-null    int32   
 10  CD_REFNIS_mun       794 non-null    Int64   
 11  Communes            794 non-null    string  
 12  Gemeenten           794 non-null    string  
 13  CD_REFNIS_prov      794 non-null    Int64   
 14  Provinces           794 non-null    string  
 15  Provincies          794 non-null

In [41]:
geo_stations = geo_stations.drop(columns=["geo_point_2d",
                                          "geo_shape",
                                          "index_right",
                                          "tgid",
                                          "languagestatute"
                                          ]
                                          )

In [42]:
geo_stations = geo_stations.rename(columns={
    "ptcarid" : "Ptcar_ID",
    "longnamefrench" : "Station_Name_French",
    "longnamedutch" : "Station_Name_Dutch",
    "class_en" : "Station_Class",
    "CD_REFNIS_mun" : "REFNIS_Municipality",
    "CD_REFNIS_prov" : "REFNIS_Province",
    "CD_REFNIS_region" : "REFNIS_Region"
})

In [43]:
geo_stations = geo_stations.astype({
    "Ptcar_ID" : "Int64",
    "Station_Name_French" : "string",
    "Station_Name_Dutch" : "string",
    "Station_Class" : "string"
})

In [44]:
geo_stations.isnull().sum()

Ptcar_ID               0
Station_Name_French    0
Station_Name_Dutch     0
Station_Class          0
geometry               0
REFNIS_Municipality    0
Communes               0
Gemeenten              0
REFNIS_Province        0
Provinces              0
Provincies             0
REFNIS_Region          0
Regions                0
Gewesten               0
Population_density     0
dtype: int64

### 1.3.6. Extracting `lon` and `lat` from `geometry`

In [45]:
geo_stations["lon"] = geo_stations.geometry.x
geo_stations["lat"] = geo_stations.geometry.y

In [46]:
geo_stations = geo_stations.drop(columns="geometry")

In [47]:
geo_stations["lon"] = geo_stations["lon"].astype("Float64")
geo_stations["lat"] = geo_stations["lat"].astype("Float64")

## 1.4. Refining `Dim_Station`

### 1.4.1. Binning `Population_density` into `Population_density_category`

* `Population_density_category` is calculated from `Population_density` to enable filtering on population density categories. 

* Three classes are created using a simplified version of the Statbel and EuroStat *DEGURBA* standard: `"cities"`, `"towns_suburbs"`, and `"rural"`.  
The following rules are applied : 

| Rule                         | Class                | Description                                             |
|------------------------------|----------------------|---------------------------------------------------------|
| `population_density` >= 1500 | `"cities"`           | Urban centers with high-density population              |
| `population_density` >= 300  | `"towns_suburbs"`    | Urban areas with moderate-density population            |
| `population_density` < 300   | `"rural"`            | Rural or small urban areas with low-density population  |

* Once `Population_density_category` is created, `Population_density` is dropped, as no calculations on a `Float64` column will be computed on this dimension.

In [48]:
geo_stations["Population_Density_Category"] = "rural"

geo_stations.loc[
    geo_stations["Population_density"] >= 300,
    "Population_Density_Category"
] = "towns_suburbs"

geo_stations.loc[
    geo_stations["Population_density"] >= 1500,
    "Population_Density_Category"
] = "cities"

In [49]:
geo_stations["Population_Density_Category"] = geo_stations["Population_Density_Category"].astype("string")

In [50]:
geo_stations = geo_stations.drop(columns=["Population_density"])

### 1.4.2. Adding Orphan Station Entries Manually

* As established in the *02_04_profiling_and_cleaning_punctuality* notebook: 
    * A `Station_status` column is added with the fixed `"active"` value for all entries in `geo_stations`.

    * Five orphan stations present in `punctuality` but unmatched in `stations_cleaned` are manually added without geographic coordinates and the fixed `"unmatched"` value in `Station_status` (see **NOTE 1** in the introduction of this notebook.)

* The result of the concatenation between `geo_stations` and these orphan stations is named `Dim_Station`.

In [51]:
geo_stations["Station_Status"] = "active"

In [52]:
orphan_stations = pd.DataFrame([
    {"Ptcar_ID": 125, "Station_Name_French": "BAULERS", 
     "Station_Name_Dutch": "BAULERS", "Station_Class": "Service installation", 
     "Station_Status" : "unmatched"},
    {"Ptcar_ID": 864, "Station_Name_French": "MORTSEL (former MORTSEL-DEURNESTEENWEG)", 
     "Station_Name_Dutch": "MORTSEL (former MORTSEL-DEURNESTEENWEG)", "Station_Class": "Stop in open track", 
     "Station_Status" : "unmatched"},
    {"Ptcar_ID": 2222, "Station_Name_French": "OLEN-SAS", 
     "Station_Name_Dutch": "OLEN-SAS", "Station_Class": "Station", "Station_Status" : "unmatched"},
    {"Ptcar_ID": 2291, "Station_Name_French": "ANTWERPEN-LINKEROEVER", 
     "Station_Name_Dutch": "ANTWERPEN-LINKEROEVER", "Station_Class": "Stop in open track", 
     "Station_Status" : "unmatched"},
    {"Ptcar_ID": 2300, "Station_Name_French": "BAASRODE-WIJKSPOREN", 
     "Station_Name_Dutch": "BAASRODE-WIJKSPOREN", "Station_Class": "Unknown", "Station_Status" : "unmatched"},
])

Dim_Station = pd.concat([geo_stations, orphan_stations], ignore_index=True)

In [53]:
Dim_Station["Station_Status"] = Dim_Station["Station_Status"].astype("string")

### 1.4.3. Adding Surrogate Key

In [54]:
Dim_Station["SK_Station"] = Dim_Station.index + 1

### 1.4.3. Reorganising Column Order

* Columns are reorganised to improve clarity.

In [55]:
Dim_Station.columns

Index(['Ptcar_ID', 'Station_Name_French', 'Station_Name_Dutch',
       'Station_Class', 'REFNIS_Municipality', 'Communes', 'Gemeenten',
       'REFNIS_Province', 'Provinces', 'Provincies', 'REFNIS_Region',
       'Regions', 'Gewesten', 'lon', 'lat', 'Population_Density_Category',
       'Station_Status', 'SK_Station'],
      dtype='str')

In [56]:
Dim_Station = (Dim_Station[[
                "SK_Station", "Ptcar_ID", "Station_Name_French", "Station_Name_Dutch", "Station_Class", 
                "Station_Status", "REFNIS_Municipality", "Communes", "Gemeenten", 
                "REFNIS_Province", "Provinces", "Provincies", "REFNIS_Region", "Regions", "Gewesten", 
                "Population_Density_Category", "lon", "lat"
                ]])

In [57]:
Dim_Station.info()

<class 'pandas.DataFrame'>
RangeIndex: 799 entries, 0 to 798
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   SK_Station                   799 non-null    int64  
 1   Ptcar_ID                     799 non-null    Int64  
 2   Station_Name_French          799 non-null    string 
 3   Station_Name_Dutch           799 non-null    string 
 4   Station_Class                799 non-null    string 
 5   Station_Status               799 non-null    string 
 6   REFNIS_Municipality          794 non-null    Int64  
 7   Communes                     794 non-null    string 
 8   Gemeenten                    794 non-null    string 
 9   REFNIS_Province              794 non-null    Int64  
 10  Provinces                    794 non-null    string 
 11  Provincies                   794 non-null    string 
 12  REFNIS_Region                794 non-null    Int64  
 13  Regions                      79

In [58]:
Dim_Station["Ptcar_ID"].duplicated().sum()

np.int64(0)

## 1.5. Exporting to Gold Layer

The **station dimension** is now ready to be exported to the **processed directory**, as `Dim_Station.parquet`, before being loaded to the SQL data warehouse.

In [59]:
Dim_Station.to_parquet(PATH_PROCESSED / "Dim_Station.parquet")

In [60]:
df_to_check = pd.read_parquet(PATH_PROCESSED / "Dim_Station.parquet")
df_to_check.head()

,SK_Station,Ptcar_ID,Station_Name_French,Station_Name_Dutch,Station_Class,Station_Status,REFNIS_Municipality,Communes,Gemeenten,REFNIS_Province,Provinces,Provincies,REFNIS_Region,Regions,Gewesten,Population_Density_Category,lon,lat
0,1,443,GENK-ZUID-L.O.-FORD,GENK-ZUID-L.O.-FORD,Station,active,71016,Genk,Genk,70000,Province du Limbourg,Provincie Limburg,2000,Région flamande,Vlaams Gewest,towns_suburbs,5.497283,50.923363
1,2,275,CHARLEROI-A.T.,CHARLEROI-A.T.,Service installation,active,52011,Charleroi,Charleroi,50000,Province du Hainaut,Provincie Henegouwen,3000,Région wallonne,Waals Gewest,cities,4.458827,50.400574
2,3,1888,WONDELGEM-BUNDEL,WONDELGEM-BUNDEL,Station,active,44021,Gand,Gent,40000,Province de Flandre orientale,Provincie Oost-Vlaanderen,2000,Région flamande,Vlaams Gewest,cities,3.718856,51.085559
3,4,2155,ZEEBRUGGE-BRAM-TANKINST.,ZEEBRUGGE-BRAM-TANKINST.,Service installation,active,31005,Bruges,Brugge,30000,Province de Flandre occidentale,Provincie West-Vlaanderen,2000,Région flamande,Vlaams Gewest,towns_suburbs,3.24638,51.306214
4,5,2154,ZEEBRUGGE-BRAM-DSP880,ZEEBRUGGE-BRAM-DSP880,Station,active,31005,Bruges,Brugge,30000,Province de Flandre occidentale,Provincie West-Vlaanderen,2000,Région flamande,Vlaams Gewest,towns_suburbs,3.249265,51.304315


In [61]:
print(df_to_check.shape)
df_to_check.dtypes

(799, 18)


SK_Station                       int64
Ptcar_ID                         Int64
Station_Name_French             string
Station_Name_Dutch              string
Station_Class                   string
Station_Status                  string
REFNIS_Municipality              Int64
Communes                        string
Gemeenten                       string
REFNIS_Province                  Int64
Provinces                       string
Provincies                      string
REFNIS_Region                    Int64
Regions                         string
Gewesten                        string
Population_Density_Category     string
lon                            Float64
lat                            Float64
dtype: object